# Café, papa y valor agregado agropecuario en Nariño, 2012–2023
### ¿Cantidades o precios? — Código reproducible del primer borrador

Seminario de Investigación Regional II · Programa de Economía · Universidad de Nariño

Repositorio: https://github.com/santacruzdavid1201-web/cafe-papa-narino

---
**Cómo ejecutarlo:** en Colab, menú **Entorno de ejecución → Ejecutar todas** (≈ 2 minutos). No hay que subir archivos: el notebook descarga cada fuente oficial y, si una fuente rechaza la conexión, usa la copia guardada en el repositorio de GitHub. Al final, la celda de la sección 9 abre la **aplicación interactiva**.

**Estructura:** 0. Configuración · 1. Fuentes y descarga · 2. Lectura y armonización · 3. Bases de análisis · 4. Descriptivos · 5. Descomposición cantidad–precio · 6. Respuesta de la oferta (panel) · 7. Contexto departamental · 8. Cifras clave · 9. Aplicación interactiva · 10. Exportar

## 0. Configuración

In [ ]:
import sys, subprocess, importlib.util, warnings
EN_COLAB = "google.colab" in sys.modules
for paquete, modulo in [("linearmodels", "linearmodels"), ("xlrd", "xlrd"), ("openpyxl", "openpyxl"),
                        ("geopandas", "geopandas"), ("plotly", "plotly"), ("ipywidgets", "ipywidgets")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=False)

import re, io, json, zipfile, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib as mpl
import statsmodels.api as sm
from scipy import stats
from linearmodels.panel import PanelOLS
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

REPO = "santacruzdavid1201-web/cafe-papa-narino"
RAW_GITHUB = f"https://raw.githubusercontent.com/{REPO}/main/datos/raw"
BASE = Path.cwd() if (Path.cwd() / "datos").exists() else Path("/content/cafe-papa-narino" if EN_COLAB else Path.cwd() / "cafe-papa-narino")
RAW, FIG, TAB = BASE / "datos" / "raw", BASE / "salidas" / "figuras", BASE / "salidas" / "tablas"
for d in (RAW, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", BASE)

ANIO_INI, ANIO_FIN = 2012, 2023        # periodo de estudio
FACTOR_CPS = 1.25                      # 1 kg de café verde ≈ 1.25 kg de café pergamino seco (CPS)

# Paleta validada (daltonismo) y estilo común
C_CAFE, C_PAPA, C_CRIOLLA, C_VA, C_GRIS, C_TXT = "#8a5a2b", "#2a78d6", "#eb6834", "#4b4a46", "#b9b8b2", "#52514e"
COLOR = {"Café": C_CAFE, "Papa": C_PAPA, "Papa criolla": C_CRIOLLA}
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.family": "DejaVu Sans", "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984",
    "axes.labelcolor": C_TXT, "xtick.color": C_TXT, "ytick.color": C_TXT,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "axes.titlesize": 9.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2})

def guardar_fig(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", bbox_inches="tight", facecolor="white")
    plt.show()

def guardar_tabla(df, nombre):
    df.to_excel(TAB / f"{nombre}.xlsx")
    df.to_csv(TAB / f"{nombre}.csv", encoding="utf-8-sig")

def norm(t):
    """Mayúsculas sin tildes ni espacios dobles: permite cruzar nombres entre fuentes."""
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"\s+", " ", t).strip()

## 1. Fuentes oficiales y descarga
Para cada fuente: (1) se intenta la URL oficial y se aplica el filtro a Nariño; (2) si falla, se usa la copia del repositorio en GitHub (`datos/raw/`), que es el mismo archivo oficial ya filtrado con la misma función. La tabla final indica qué fuente se usó.

In [ ]:
CAB = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/130.0 Safari/537.36"}
SIPSA_S = "https://www.dane.gov.co/files/investigaciones/agropecuario/sipsa/series-historicas"
SIPSA_O = "https://www.dane.gov.co/files/operaciones/SIPSA"
SIPSA_URLS = [f"{SIPSA_S}/series-historicas-precios-mayoristas.xlsx"] + \
             [f"{SIPSA_S}/series-historicas-precios-mayoristas-{a}.xlsx" for a in range(2018, 2023)] + \
             [f"{SIPSA_O}/anex-SIPSA-SerieHistoricaMayorista-Dic2023.xlsx", f"{SIPSA_O}/anex-SIPSA-SerieHistoricaMayorista-2024.xlsx"]

def descargar(url, timeout=180):
    r = requests.get(url, headers=CAB, timeout=timeout)
    if r.status_code != 200 or len(r.content) < 2000 or r.content.lstrip().startswith(b"{\"code"):
        raise IOError(f"HTTP {r.status_code}")
    return r.content

def filtrar_eva(contenido):
    """EVA (datos.gov.co): filas de Nariño con café o papa (todas las columnas originales)."""
    d = pd.read_csv(io.BytesIO(contenido), dtype=str)
    cols = {c: norm(c) for c in d.columns}
    dep = next(c for c, n in cols.items() if n == "DEPARTAMENTO")
    cul = next(c for c, n in cols.items() if n == "CULTIVO")
    return d[d[dep].map(norm).str.startswith("NARI") & d[cul].map(norm).isin(["CAFE", "PAPA"])]

def filtrar_sipsa(contenido, origen):
    """SIPSA (DANE): precios mayoristas mensuales de papa en Pasto e Ipiales (todas las hojas)."""
    salida = []
    for hoja, df in pd.read_excel(io.BytesIO(contenido), sheet_name=None, header=None).items():
        fila = next((i for i in range(min(40, len(df))) if "Producto" in [str(v).strip() for v in df.iloc[i].values]), None)
        if fila is None:
            continue
        d = df.iloc[fila + 1:].copy()
        d.columns = [str(c).strip() for c in df.iloc[fila]]
        merc = next(c for c in d.columns if c.startswith(("Mercado", "Fuente")))
        prec = next(c for c in d.columns if c.startswith("Precio"))
        d = d.rename(columns={merc: "Mercado", prec: "Precio"})[["Fecha", "Grupo", "Producto", "Mercado", "Precio"]]
        d = d[d.Producto.astype(str).str.startswith("Papa ") & d.Mercado.astype(str).str.contains("Pasto|Ipiales")]
        salida.append(d.assign(archivo=origen))
    return pd.concat(salida, ignore_index=True)

FUENTES = [  # (archivo en datos/raw, URL oficial, función que produce el archivo, descripción)
    ("eva_2007_2018_narino.csv", "https://www.datos.gov.co/api/views/2pnw-mmge/rows.csv?accessType=DOWNLOAD",
     lambda: filtrar_eva(descargar("https://www.datos.gov.co/api/views/2pnw-mmge/rows.csv?accessType=DOWNLOAD", 600)),
     "MinAgricultura/UPRA – Evaluaciones Agropecuarias Municipales (EVA) 2007–2018, API datos.gov.co"),
    ("eva_2019_2025_narino.csv", "https://www.datos.gov.co/api/views/uejq-wxrr/rows.csv?accessType=DOWNLOAD",
     lambda: filtrar_eva(descargar("https://www.datos.gov.co/api/views/uejq-wxrr/rows.csv?accessType=DOWNLOAD", 600)),
     "MinAgricultura/UPRA – EVA 2019–2025, API datos.gov.co"),
    ("sipsa_papa_pasto_ipiales.csv", "; ".join(SIPSA_URLS),
     lambda: pd.concat([filtrar_sipsa(descargar(u), u.rsplit("/", 1)[1]) for u in SIPSA_URLS], ignore_index=True),
     "DANE – SIPSA, series históricas de precios mayoristas 2013–2024 (papa, Pasto e Ipiales)"),
    ("fnc_precios_area_produccion.xlsx", "https://federaciondecafeteros.org/wp-content/uploads/2026/09/Precios-area-y-produccion-de-cafe-Noviembre-2026.xlsx",
     lambda: descargar("https://federaciondecafeteros.org/wp-content/uploads/2026/09/Precios-area-y-produccion-de-cafe-Noviembre-2026.xlsx"),
     "FNC – Precios, área y producción de café (precio interno mensual, precio externo, área por departamento)"),
    ("pib_actividad.xlsx", "https://www.dane.gov.co/files/operaciones/PIB/anex-PIBDep-Activecono-2025pr.xlsx",
     lambda: descargar("https://www.dane.gov.co/files/operaciones/PIB/anex-PIBDep-Activecono-2025pr.xlsx"),
     "DANE – Valor agregado por actividad económica y departamento, base 2015, 2005–2025pr"),
    ("mapa_municipios_narino.geojson", "https://raw.githubusercontent.com/caticoa3/colombia_mapa/master/co_2018_MGN_MPIO_POLITICO.geojson",
     lambda: json.dumps({"type": "FeatureCollection", "features": [f for f in json.loads(descargar(
         "https://raw.githubusercontent.com/caticoa3/colombia_mapa/master/co_2018_MGN_MPIO_POLITICO.geojson"))["features"]
         if str(f["properties"]["DPTO_CCDGO"]) == "52"]}).encode(),
     "Límites municipales MGN-DANE 2018 (GeoJSON, repositorio público caticoa3/colombia_mapa), filtrado a Nariño"),
]

INTENTAR_OFICIAL = True     # False = usar directamente la copia del repositorio (réplica exacta del PDF)

def obtener(archivo, producir):
    destino = RAW / archivo
    if INTENTAR_OFICIAL:
        try:
            dato = producir()
            if isinstance(dato, pd.DataFrame):
                dato.to_csv(destino, index=False, encoding="utf-8")
            else:
                destino.write_bytes(dato)
            return "Fuente oficial"
        except Exception as e:
            motivo = type(e).__name__
    else:
        motivo = "desactivado"
    if destino.exists():
        return f"Copia local ({motivo})"
    try:
        destino.write_bytes(descargar(f"{RAW_GITHUB}/{archivo}"))
        return f"Copia en GitHub ({motivo})"
    except Exception as e:
        raise FileNotFoundError(f"{archivo}: sin fuente oficial ({motivo}) ni copia en GitHub ({type(e).__name__})")

estado = pd.DataFrame([(a, obtener(a, f), d) for a, _, f, d in FUENTES], columns=["archivo", "fuente usada", "descripción"])
estado["KB"] = [round((RAW / a).stat().st_size / 1024) for a in estado.archivo]
display(estado)

## 2. Lectura y armonización
* **Café (EVA):** hasta 2018 se reporta en café verde equivalente y desde 2019 en pergamino seco; se lleva todo a pergamino seco (CPS) con el factor 1.25, que es la unidad del precio interno de la FNC.
* **Papa (EVA):** se suman los semestres A y B. En 2018 la EVA solo trae el semestre A, por lo que 2018 queda como dato faltante para la papa.
* **Quiebre metodológico:** en 2019 la EVA cambia de metodología (sube el nivel de la papa). Las variaciones 2018→2019 de cantidades se excluyen y los modelos incluyen una dummy D2019.

In [ ]:
def leer_eva():
    partes = []
    for archivo in ["eva_2007_2018_narino.csv", "eva_2019_2025_narino.csv"]:
        d = pd.read_csv(RAW / archivo, dtype=str)
        d.columns = [norm(c) for c in d.columns]
        col = lambda *p: next(c for c in d.columns if any(c.startswith(x) for x in p))
        out = pd.DataFrame({
            "cod_mpio": d[col("COD. MUN", "CODIGO DANE MUNICIPIO")].str.zfill(5),
            "municipio": d[col("MUNICIPIO")].map(norm),
            "cultivo": d[col("CULTIVO")].map(norm),
            "desag": d[col("DESAGREGACION")].map(norm),
            "anio": pd.to_numeric(d[col("ANO")]),
            "periodo": d[col("PERIODO")],
            "area_sem": pd.to_numeric(d[col("AREA SEMBRADA")].str.replace(",", ""), errors="coerce"),
            "area_cos": pd.to_numeric(d[col("AREA COSECHADA")].str.replace(",", ""), errors="coerce"),
            "prod": pd.to_numeric(d[col("PRODUCCION")].str.replace(",", ""), errors="coerce"),
            "estado": d[col("ESTADO FISICO")].map(norm)})
        partes.append(out)
    e = pd.concat(partes, ignore_index=True)
    e["producto"] = np.where(e.cultivo == "CAFE", "Café", np.where(e.desag.str.contains("CRIOLLA"), "Papa criolla", "Papa"))
    verde = e.estado.str.contains("VERDE")
    for c in ["prod"]:
        e.loc[(e.producto == "Café") & verde, c] *= FACTOR_CPS
    a = e.groupby(["cod_mpio", "producto", "anio"], as_index=False).agg(
        municipio=("municipio", "first"), area_sem=("area_sem", "sum"), area_cos=("area_cos", "sum"),
        prod=("prod", "sum"), n_periodos=("periodo", "nunique"))
    # 2018: la EVA solo publica el semestre A de la papa → año incompleto
    a.loc[a.producto.str.startswith("Papa") & (a.anio == 2018), ["area_sem", "area_cos", "prod"]] = np.nan
    return a

eva = leer_eva()
NOMBRE_MPIO = eva.groupby("cod_mpio").municipio.first().str.title().to_dict()
print("EVA:", eva.cod_mpio.nunique(), "municipios,", eva.anio.min(), "–", eva.anio.max())
display(eva.groupby(["anio", "producto"])["prod"].sum().unstack().loc[2010:].round(0))

def leer_precios_papa():
    s = pd.read_csv(RAW / "sipsa_papa_pasto_ipiales.csv")
    s["Fecha"] = pd.to_datetime(s.Fecha, errors="coerce")
    s["Precio"] = pd.to_numeric(s.Precio, errors="coerce")
    s = s.dropna(subset=["Fecha", "Precio"])
    s["anio"] = s.Fecha.dt.year
    s["plaza"] = np.where(s.Mercado.str.startswith("Pasto"), "Pasto", "Ipiales")
    s["producto"] = np.where(s.Producto.str.contains("criolla"), "Papa criolla",
                             np.where(s.Producto.isin(["Papa capira", "Papa única"]), "Papa", None))
    s = s.dropna(subset=["producto"])
    mensual = s.groupby(["plaza", "producto", "Fecha"], as_index=False).Precio.mean()
    mensual["anio"] = mensual.Fecha.dt.year
    return mensual

papa_m = leer_precios_papa()
print("SIPSA papa:", papa_m.Fecha.min().date(), "–", papa_m.Fecha.max().date())

def hoja_fnc(nombre, fila_ini=6):
    d = pd.read_excel(RAW / "fnc_precios_area_produccion.xlsx", nombre, header=None)
    return d

pi = hoja_fnc("2. Precio Interno Mensual")
cafe_m = pd.DataFrame({"Fecha": pd.to_datetime(pi[3], errors="coerce"), "Precio": pd.to_numeric(pi[4], errors="coerce")}).dropna()
cafe_m["Precio"] = cafe_m.Precio / 125                 # $ por kg de pergamino seco
cafe_m["anio"] = cafe_m.Fecha.dt.year
pe = hoja_fnc("4. Precio Ex_Dock Anual Civil")
precio_ext = pd.DataFrame({"anio": pd.to_numeric(pe[3], errors="coerce"), "p_ext": pd.to_numeric(pe[4], errors="coerce")}).dropna()
precio_ext["anio"] = precio_ext.anio.astype(int)
ar = hoja_fnc("7. Área cult. dep. producto")
fila_n = ar.index[ar[1].astype(str).str.startswith("Nari")][0]
area_fnc = pd.DataFrame({"anio": [int(str(v)[:4]) for v in ar.iloc[5, 2:].dropna()],
                         "area_fnc_miles_ha": pd.to_numeric(ar.iloc[fila_n, 2:2 + ar.iloc[5, 2:].notna().sum()].values, errors="coerce")})

# Valor agregado agropecuario de Nariño (DANE, base 2015)
def leer_va(hoja):
    df = pd.read_excel(RAW / "pib_actividad.xlsx", hoja, header=None)
    actividad = None
    for i in range(df.shape[0]):
        c0, c1 = df.iat[i, 0], df.iat[i, 1]
        if isinstance(c0, str) and pd.isna(c1) and c0.strip().startswith("Agricultura"):
            actividad = i
        if actividad is not None and str(c1).strip().upper().startswith("DEPARTAMENTO"):
            enc = df.iloc[i]
            anios = {}
            for j in range(2, df.shape[1]):
                if pd.isna(enc[j]) or not str(enc[j])[:4].isdigit():
                    break
                anios[j] = int(str(enc[j])[:4])
            fila = next(k for k in range(i + 1, i + 40) if str(df.iat[k, 0]).strip() in ("52", "52.0"))
            return pd.Series({a: float(df.iat[fila, j]) for j, a in anios.items()})
va = pd.DataFrame({"va_nom": leer_va("Cuadro 1"), "va_real": leer_va("Cuadro 2")})
va.index.name = "anio"
va["deflactor"] = 100 * va.va_nom / va.va_real          # deflactor implícito del VA agropecuario de Nariño (2015 = 100)
va = va.reset_index()
print("VA agropecuario Nariño:", va.anio.min(), "–", va.anio.max())

## 3. Bases de análisis
**Ingreso bruto** (valor de la producción) = toneladas × precio, la misma lógica con que el DANE valora la producción agrícola. Precios reales en pesos de 2015 con el deflactor implícito del VA agropecuario de Nariño.
* **Panel municipal:** municipio × producto × año.
* **Serie departamental:** suma de municipios, por producto.

In [ ]:
def precios_anuales(plaza_papa="Pasto"):
    p = papa_m[papa_m.plaza == plaza_papa].groupby(["producto", "anio"], as_index=False).Precio.mean()
    c = cafe_m.groupby("anio", as_index=False).Precio.mean().assign(producto="Café")
    p = pd.concat([p, c], ignore_index=True).rename(columns={"Precio": "p_nom"})
    p = p.merge(va[["anio", "deflactor"]], on="anio", how="left")
    p["p_real"] = 100 * p.p_nom / p.deflactor
    return p[["producto", "anio", "p_nom", "p_real"]]

def construir_panel(plaza_papa="Pasto"):
    pr = precios_anuales(plaza_papa)
    d = eva.merge(pr, on=["producto", "anio"], how="left").sort_values(["cod_mpio", "producto", "anio"])
    d["rend"] = d["prod"] / d.area_cos
    d["ib_nom"] = d["prod"] * 1000 * d.p_nom / 1e6          # millones de pesos
    d["ib_real"] = d["prod"] * 1000 * d.p_real / 1e6
    for v in ["prod", "area_cos", "area_sem", "rend", "ib_real", "p_real"]:
        d[f"ln_{v}"] = np.log(d[v].where(d[v] > 0))
    g = d.groupby(["cod_mpio", "producto"])
    for v in ["prod", "area_cos", "rend", "ib_real", "p_real"]:
        d[f"d_{v}"] = g[f"ln_{v}"].diff() * 100
        # se excluyen variaciones que cruzan el cambio metodológico de 2019 o años no consecutivos
        salto = (g.anio.diff() != 1) | (d.anio == 2019)
        if v != "p_real":
            d.loc[salto, f"d_{v}"] = np.nan
    for k in (1, 2):
        d[f"ln_p_real_l{k}"] = g.ln_p_real.shift(k).where(g.anio.diff(k) == k)
    d["D2019"] = (d.anio >= 2019).astype(float)
    d["tendencia"] = d.anio - ANIO_INI
    return d

panel = construir_panel("Pasto")
panel_ipiales = construir_panel("Ipiales")
est = panel[panel.anio.between(ANIO_INI, ANIO_FIN)].copy()

def serie_departamental(d):
    s = d.groupby(["producto", "anio"], as_index=False)[["area_sem", "area_cos", "prod", "ib_nom", "ib_real"]].sum(min_count=1)
    s = s.merge(precios_anuales(), on=["producto", "anio"], how="left")
    s["rend"] = s["prod"] / s.area_cos
    s.loc[s.p_nom.isna(), ["ib_nom", "ib_real"]] = np.nan
    return s

dep = serie_departamental(panel)
print(f"Panel municipal 2012–2023: {est.cod_mpio.nunique()} municipios, {len(est)} observaciones municipio-producto-año")
display(dep[dep.anio.between(ANIO_INI, ANIO_FIN)].pivot(index="anio", columns="producto", values=["prod", "p_real", "ib_real"]).round(0))

## 4. Descriptivos (objetivo 1)
### Tabla 2. Operacionalización de variables

In [ ]:
def cobertura(v, prod=None):
    d = est if prod is None else est[est.producto == prod]
    d = d.dropna(subset=[v])
    return f"{d.anio.min()}–{d.anio.max()} · {d.cod_mpio.nunique()} mpios."

tabla2 = pd.DataFrame([
    ("Producción", "EVA (MinAgricultura–UPRA)", "t (café: pergamino seco)", "Suma de semestres; café verde × 1.25 hasta 2018", cobertura("prod")),
    ("Área cosechada", "EVA", "ha", "Suma de semestres", cobertura("area_cos")),
    ("Rendimiento", "EVA", "t/ha", "Producción / área cosechada", cobertura("rend")),
    ("Precio del café", "FNC, precio interno de referencia", "$/kg CPS", "Promedio anual de la serie mensual ÷ 125 kg", "2012–2023 · Nariño"),
    ("Precio de la papa", "DANE-SIPSA, Pasto (El Potrerillo)", "$/kg mayorista", "Promedio de Capiro y Única; criolla aparte", "2013–2023 · Nariño"),
    ("Precio real", "Precio nominal y deflactor del VA agropecuario", "$ de 2015/kg", "Nominal / deflactor implícito × 100", "2012–2023"),
    ("Ingreso bruto", "Producción × precio", "Millones de $ de 2015", "ln y Δln × 100", cobertura("ib_real")),
    ("VA agropecuario", "DANE, cuentas departamentales base 2015", "Miles de millones de $", "Corriente y volumen encadenado", "2005–2025pr · Nariño"),
], columns=["Variable", "Fuente", "Unidad", "Transformación", "Cobertura"]).set_index("Variable")
guardar_tabla(tabla2, "Tabla_2_operacionalizacion")
display(tabla2)

### Tabla 5. Café y papa en Nariño, 2012–2023

In [ ]:
def resumen_producto(p):
    s = dep[(dep.producto == p) & dep.anio.between(ANIO_INI, ANIO_FIN)].set_index("anio")
    m = est[est.producto == p]
    pm = papa_m if p != "Café" else cafe_m.assign(plaza="Pasto", producto="Café")
    pm = pm[(pm.plaza == "Pasto") & (pm.producto == p) & pm.anio.between(ANIO_INI, ANIO_FIN)]
    vol_m = np.log(pm.Precio).diff().std() * 100
    top5 = m.groupby("cod_mpio")["prod"].mean().nlargest(5).sum() / m.groupby("cod_mpio")["prod"].mean().sum() * 100
    dq = s.dropna(subset=["prod"])
    return {"Municipios productores": m[m["prod"] > 0].cod_mpio.nunique(),
            "Producción media (t)": s["prod"].mean(), "Área cosechada media (ha)": s.area_cos.mean(),
            "Rendimiento medio (t/ha)": s.rend.mean(), "Precio real medio ($ 2015/kg)": s.p_real.mean(),
            "Ingreso bruto real medio (miles de millones $ 2015)": s.ib_real.mean() / 1000,
            "Participación de los 5 mayores municipios (%)": top5,
            "Volatilidad mensual del precio (desv. Δln, %)": vol_m,
            "Desv. Δln producción anual (%)": (np.log(dq["prod"]).diff().where(dq.index.to_series().diff() == 1)).drop(index=2019, errors="ignore").std() * 100,
            "Desv. Δln precio real anual (%)": np.log(s.p_real).diff().std() * 100}
tabla5 = pd.DataFrame({p: resumen_producto(p) for p in ["Café", "Papa", "Papa criolla"]})
guardar_tabla(tabla5.round(2), "Tabla_5_descriptivos")
display(tabla5.round(2))

### Figura 1. Producción media por municipio, 2012–2023

In [ ]:
import geopandas as gpd
mapa = gpd.read_file(RAW / "mapa_municipios_narino.geojson")
mapa["cod_mpio"] = mapa.MPIO_CCNCT.astype(str).str.zfill(5)
prom = est.groupby(["cod_mpio", "producto"])["prod"].mean().unstack()
mapa = mapa.merge(prom, left_on="cod_mpio", right_index=True, how="left")
fig, ejes = plt.subplots(1, 2, figsize=(9.0, 4.4), gridspec_kw={"wspace": 0.35})
for ax, p, cmap in [(ejes[0], "Café", "YlOrBr"), (ejes[1], "Papa", "Blues")]:
    mapa.plot(color="#f1efe8", edgecolor="white", linewidth=0.3, ax=ax)
    mapa[mapa[p] > 0].plot(column=p, cmap=cmap, edgecolor="white", linewidth=0.3, ax=ax, legend=True,
                           legend_kwds={"shrink": 0.5, "orientation": "horizontal", "pad": 0.01, "label": "toneladas por año"})
    top = mapa.nlargest(3, p)
    xmin, ymin, xmax, ymax = mapa.total_bounds
    for k, (_, r) in enumerate(top.iterrows()):
        x, y = r.geometry.representative_point().coords[0]
        ax.annotate(r.MPIO_CNMBR.title(), (x, y), xytext=(xmax + 0.05, ymax - 0.35 - 0.3 * k), fontsize=6.8,
                    fontweight="bold", ha="left", va="center", annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", color="#333333", lw=0.6))
    ax.set_title(f"{'a' if p == 'Café' else 'b'}) {p}: producción media anual", fontsize=8.5)
    ax.set_axis_off()
    cb = fig.axes[-1]
    cb.tick_params(labelsize=7)
    cb.xaxis.label.set_size(7.5)
guardar_fig(fig, "Figura_1_mapas_produccion")

### Figura 2. Producción, precio real e ingreso bruto en Nariño

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(10.2, 3.0))
idx = lambda s: 100 * s / s.loc[2013]
for p in ["Café", "Papa", "Papa criolla"]:
    s = dep[(dep.producto == p)].set_index("anio").loc[ANIO_INI:ANIO_FIN]
    q = s["prod"]
    ejes[0].plot(idx(q).loc[:2017], color=COLOR[p], label=p)
    ejes[0].plot(idx(q).loc[2019:], color=COLOR[p])
    ejes[1].plot(idx(s.p_real).dropna(), color=COLOR[p], label=p)
    ejes[2].plot(s.ib_real / 1000, color=COLOR[p], marker="o", ms=3, label=p)
for ax in ejes[:2]:
    ax.axhline(100, color="#8a8984", lw=0.7)
ejes[0].axvline(2018.5, color="#8a8984", lw=0.8, ls=":")
ejes[0].text(2018.6, ejes[0].get_ylim()[1] * 0.97, "cambio\nmetodológico\nEVA", fontsize=6.5, color=C_TXT, va="top")
ejes[0].set_title("a) Producción (índice 2013 = 100)")
ejes[1].set_title("b) Precio real (índice 2013 = 100)")
ejes[2].set_title("c) Ingreso bruto real (miles de millones $ 2015)")
ejes[1].legend(fontsize=7, loc="upper left")
fig.tight_layout()
guardar_fig(fig, "Figura_2_series")

## 5. Descomposición del ingreso bruto: cantidad frente a precio (objetivos 2 y 4)
$\Delta\ln IB_t = \Delta\ln A_t + \Delta\ln R_t + \Delta\ln P_t$ (área cosechada, rendimiento y precio real). La participación de cada componente en la varianza es $s_X=\mathrm{cov}(\Delta\ln IB,\Delta\ln X)/\mathrm{var}(\Delta\ln IB)$ y las tres suman 1. Se calcula para el agregado departamental y para el panel municipal (con intervalos bootstrap por municipio).

In [ ]:
COMP = {"d_area_cos": "Área", "d_rend": "Rendimiento", "d_p_real": "Precio"}

def participaciones(d):
    d = d.dropna(subset=["d_ib_real"] + list(COMP))
    v = d.d_ib_real.var()
    return pd.Series({et: np.cov(d.d_ib_real, d[c])[0, 1] / v for c, et in COMP.items()} | {"n": len(d)})

def depto_cambios(p):
    s = dep[dep.producto == p].set_index("anio").loc[ANIO_INI:ANIO_FIN].copy()
    out = pd.DataFrame(index=s.index)
    for v, c in [("ib_real", "d_ib_real"), ("area_cos", "d_area_cos"), ("rend", "d_rend"), ("p_real", "d_p_real")]:
        out[c] = np.log(s[v]).diff() * 100
    out.loc[out.index == 2019, ["d_ib_real", "d_area_cos", "d_rend"]] = np.nan
    return out

rng = np.random.default_rng(20261109)
def bootstrap_mpios(d, B=999):
    mps = d.cod_mpio.unique()
    res = []
    for _ in range(B):
        muestra = pd.concat([d[d.cod_mpio == m] for m in rng.choice(mps, len(mps))])
        res.append(participaciones(muestra)[list(COMP.values())])
    return pd.DataFrame(res).quantile([0.025, 0.975])

filas6 = []
for p in ["Café", "Papa", "Papa criolla"]:
    sd = participaciones(depto_cambios(p))
    filas6.append({"Producto": p, "Nivel": "Departamento", **sd.to_dict()})
    dm = est[(est.producto == p) & (est["prod"] > 0)]
    sm_ = participaciones(dm)
    ci = bootstrap_mpios(dm.dropna(subset=["d_ib_real"] + list(COMP)), B=499)
    filas6.append({"Producto": p, "Nivel": "Municipios (panel)", **sm_.to_dict(),
                   **{f"IC95 {k}": f"[{ci.loc[0.025, k]:.2f}, {ci.loc[0.975, k]:.2f}]" for k in COMP.values()}})
tabla6 = pd.DataFrame(filas6).set_index(["Producto", "Nivel"])
tabla6["n"] = tabla6.n.astype(int)
guardar_tabla(tabla6.round(3), "Tabla_6_descomposicion")
display(tabla6.round(3))

### Figura 3. Descomposición acumulada por subperiodo y heterogeneidad municipal

In [ ]:
def acumulado(p, a0, a1):
    s = dep[dep.producto == p].set_index("anio")
    return {"Área": np.log(s.area_cos[a1] / s.area_cos[a0]) * 100, "Rendimiento": np.log(s.rend[a1] / s.rend[a0]) * 100,
            "Precio": np.log(s.p_real[a1] / s.p_real[a0]) * 100, "Total": np.log(s.ib_real[a1] / s.ib_real[a0]) * 100}
TRAMOS = [("Café", 2012, 2018), ("Café", 2019, 2023), ("Papa", 2013, 2017), ("Papa", 2019, 2023),
          ("Papa criolla", 2013, 2017), ("Papa criolla", 2019, 2023)]
acum = pd.DataFrame([{"Producto": p, "Tramo": f"{a0}–{a1}", **acumulado(p, a0, a1)} for p, a0, a1 in TRAMOS])
guardar_tabla(acum.set_index(["Producto", "Tramo"]).round(2), "Tabla_6b_descomposicion_acumulada")

# participación del precio por municipio (municipios con ≥ 6 variaciones válidas)
part_m = (est.dropna(subset=["d_ib_real"] + list(COMP)).groupby(["producto", "cod_mpio"])
          .filter(lambda g: len(g) >= 6).groupby(["producto", "cod_mpio"]).apply(participaciones))
fig = plt.figure(figsize=(10.2, 3.8))
ax = fig.add_axes([0.06, 0.25, 0.46, 0.65])
x = np.arange(len(acum))
base_pos, base_neg = np.zeros(len(acum)), np.zeros(len(acum))
for comp, c in [("Área", "#9fc3ef"), ("Rendimiento", "#2a78d6"), ("Precio", "#eb6834")]:
    v = acum[comp].values
    ax.bar(x, np.where(v > 0, v, 0), bottom=base_pos, color=c, width=0.62, label=comp, edgecolor="white", linewidth=0.6)
    ax.bar(x, np.where(v < 0, v, 0), bottom=base_neg, color=c, width=0.62, edgecolor="white", linewidth=0.6)
    base_pos += np.where(v > 0, v, 0)
    base_neg += np.where(v < 0, v, 0)
ax.scatter(x, acum.Total, color="#111111", marker="D", s=24, zorder=3, label="Ingreso bruto (total)")
ax.set_ylim(min(base_neg.min(), acum.Total.min()) * 1.15 - 2, max(base_pos.max(), acum.Total.max()) * 1.1 + 2)
ax.axhline(0, color="#8a8984", lw=0.8)
ax.set_xticks(x, [f"{p}\n{t}" for p, t in zip(acum.Producto, acum.Tramo)], fontsize=7)
ax.set_ylabel("Cambio acumulado (100·Δln)")
ax.set_title("a) Descomposición acumulada del ingreso bruto real")
ax.legend(fontsize=6.8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.17))
ax.grid(axis="x", visible=False)
ax2 = fig.add_axes([0.59, 0.25, 0.39, 0.65])
for i, p in enumerate(["Café", "Papa", "Papa criolla"]):
    if p in part_m.index.get_level_values(0):
        v = part_m.loc[p, "Precio"].clip(-0.5, 1.5)
        ax2.scatter(v, np.full(len(v), i) + rng.uniform(-0.15, 0.15, len(v)), color=COLOR[p], s=18, alpha=0.75,
                    edgecolor="white", linewidth=0.4)
        ax2.plot([v.median()] * 2, [i - 0.3, i + 0.3], color="#111111", lw=1.6)
        d_val = tabla6.loc[(p, "Departamento"), "Precio"]
        ax2.scatter([d_val], [i], marker="*", s=110, color="#111111", zorder=3)
ax2.axvline(0.5, color="#8a8984", lw=0.8, ls=":")
ax2.set_yticks(range(3), ["Café", "Papa", "Papa criolla"])
ax2.set_xlabel("Participación del precio en la varianza del ingreso bruto")
ax2.set_title("b) Municipios (puntos), mediana (línea) y departamento (★)")
guardar_fig(fig, "Figura_3_descomposicion")

## 6. Respuesta de la producción al precio: panel municipal con efectos fijos (objetivo 3)
$$\ln A_{mt} = \alpha_m + \beta \ln P^{real}_{t-1} + \delta\, t + \theta D2019_t + \varepsilon_{mt}$$
El precio es común a todos los municipios (varía solo en el tiempo), por eso no se usan efectos fijos de año y los errores son de **Driscoll-Kraay**, robustos a autocorrelación y a dependencia transversal. β es la elasticidad de la oferta (área) al precio real del año anterior (modelo de Nerlove).

In [ ]:
def estimar_oferta(d, y="ln_area_cos", x="ln_p_real_l1", controles=("tendencia", "D2019"), cov="kernel"):
    cols = [y, x] + list(controles)
    dd = d.dropna(subset=cols)
    dd = dd[np.isfinite(dd[cols]).all(axis=1)].set_index(["cod_mpio", "anio"])
    mod = PanelOLS(dd[y], dd[[x] + list(controles)], entity_effects=True)
    if cov == "kernel":
        return mod.fit(cov_type="kernel", kernel="bartlett", bandwidth=2)
    return mod.fit(cov_type="clustered", cluster_entity=True)

def estrellas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

ofe = {p: est[(est.producto == p) & (est["prod"] > 0)] for p in ["Café", "Papa", "Papa criolla"]}
MODELOS = {
    "Café · área": estimar_oferta(ofe["Café"]),
    "Café · producción": estimar_oferta(ofe["Café"], "ln_prod"),
    "Papa · área": estimar_oferta(ofe["Papa"]),
    "Papa · producción": estimar_oferta(ofe["Papa"], "ln_prod"),
    "Papa criolla · área": estimar_oferta(ofe["Papa criolla"]),
}
ETQ = {"ln_p_real_l1": "ln precio real (t−1)", "tendencia": "Tendencia", "D2019": "D2019 (cambio EVA)"}
filas7 = {}
for n, r in MODELOS.items():
    c = {}
    for v, et in ETQ.items():
        c[et] = f"{r.params[v]:.3f}{estrellas(r.pvalues[v])}"
        c[et + " "] = f"({r.std_errors[v]:.3f})"
    c.update({"Observaciones": f"{int(r.nobs)}", "Municipios": f"{int(r.entity_info['total'])}",
              "R² within": f"{r.rsquared_within:.3f}"})
    filas7[n] = c
tabla7 = pd.DataFrame(filas7)
tabla7.loc["Efectos fijos de municipio"] = "Sí"
guardar_tabla(tabla7, "Tabla_7_oferta")
display(tabla7)
print("Errores Driscoll-Kraay (Bartlett, ancho de banda 2) entre paréntesis. *** p<0.01; ** p<0.05; * p<0.10.")

### Diagnósticos (Tabla 8, panel A) y robustez (Figura 4)

In [ ]:
def resid_matriz(res):
    e = res.resids.reset_index()
    e.columns = ["mpio", "anio", "e"]
    return e.pivot(index="anio", columns="mpio", values="e")

def pesaran_cd(E):
    T = E.notna().sum().min()
    R = E.corr(min_periods=4).values
    iu = np.triu_indices(R.shape[0], 1)
    r = R[iu][~np.isnan(R[iu])]
    N = E.shape[1]
    Tij = E.notna().astype(int).T.dot(E.notna().astype(int)).values[iu][~np.isnan(R[iu])]
    cd = np.sqrt(2 / (N * (N - 1))) * np.sum(np.sqrt(Tij) * r)
    return cd, 2 * stats.norm.sf(abs(cd)), r.mean()

def wooldridge_ar1(res):
    e = res.resids.reset_index()
    e.columns = ["mpio", "anio", "e"]
    e = e.sort_values(["mpio", "anio"])
    e["de"] = e.groupby("mpio").e.diff()
    e["de_l"] = e.groupby("mpio").de.shift(1)
    d = e.dropna()
    r = sm.OLS(d.de, d.de_l).fit(cov_type="cluster", cov_kwds={"groups": d.mpio.astype("category").cat.codes})
    F = ((r.params.iloc[0] + 0.5) / r.bse.iloc[0]) ** 2
    return F, stats.f.sf(F, 1, d.mpio.nunique() - 1), r.params.iloc[0]

def wald_het(E):
    s2 = (E ** 2).mean()
    T = E.notna().sum()
    V = ((E ** 2 - s2) ** 2).sum() / (T * (T - 1))
    ok = (T > 3) & (V > 0)
    W = (((s2 - (E ** 2).stack().mean()) ** 2) / V)[ok].sum()
    return W, stats.chi2.sf(W, ok.sum())

diag = []
for n in ["Café · área", "Papa · área"]:
    r = MODELOS[n]
    E = resid_matriz(r)
    cd, pcd, rho = pesaran_cd(E)
    F, pw, ar = wooldridge_ar1(r)
    W, ph = wald_het(E)
    diag += [(n, "F de efectos fijos", r.f_pooled.stat, r.f_pooled.pval, "H0: sin efectos de municipio"),
             (n, "Pesaran CD", cd, pcd, f"H0: independencia transversal; correlación media {rho:.2f}"),
             (n, "Wooldridge AR(1)", F, pw, f"H0: sin autocorrelación; coef. {ar:.2f} vs −0.5"),
             (n, "Wald modificado", W, ph, "H0: varianza homogénea entre municipios")]
tabla8a = pd.DataFrame(diag, columns=["Modelo", "Prueba", "Estadístico", "Valor p", "Lectura"]).set_index(["Modelo", "Prueba"])
guardar_tabla(tabla8a.round(3), "Tabla_8a_diagnosticos")
display(tabla8a.round(3))

# Robustez de la elasticidad β
ext = precio_ext.set_index("anio").p_ext
def con_precio_alt(d, p):
    d = d.copy()
    if p == "Café":
        d["ln_p_alt_l1"] = np.log(d.anio.map(lambda a: ext.get(a - 1, np.nan)))
    else:
        alt = panel_ipiales[panel_ipiales.producto == p].set_index(["cod_mpio", "anio"]).ln_p_real_l1
        d["ln_p_alt_l1"] = d.set_index(["cod_mpio", "anio"]).index.map(alt.to_dict())
    return d

def nominal_l1(d):
    d = d.copy()
    d["ln_p_nom_l1"] = d.groupby("cod_mpio").p_nom.transform(lambda s: np.log(s).shift(1))
    return d

ESPEC = []
for p in ["Café", "Papa"]:
    d = ofe[p]
    alt = "Precio alternativo (café: externo; papa: Ipiales)"
    specs = {"Base: área cosechada, Driscoll-Kraay": estimar_oferta(d),
             "Producción": estimar_oferta(d, "ln_prod"),
             "Área sembrada": estimar_oferta(d, "ln_area_sem"),
             "Precio rezagado 2 años": estimar_oferta(d, x="ln_p_real_l2"),
             alt: estimar_oferta(con_precio_alt(d, p), x="ln_p_alt_l1"),
             "Precio nominal": estimar_oferta(nominal_l1(d), x="ln_p_nom_l1"),
             "Sin 2020–2021": estimar_oferta(d[~d.anio.isin([2020, 2021])]),
             "Errores agrupados por municipio": estimar_oferta(d, cov="cluster")}
    for k, r in specs.items():
        xv = [c for c in r.params.index if c.startswith("ln_p")][0]
        ESPEC.append({"Producto": p, "Especificación": k,
                      "beta": r.params[xv], "ee": r.std_errors[xv], "p": r.pvalues[xv], "N": int(r.nobs)})
rob = pd.DataFrame(ESPEC)
guardar_tabla(rob.set_index(["Producto", "Especificación"]).round(3), "Tabla_7b_robustez")

fig, ejes = plt.subplots(1, 2, figsize=(9.6, 3.4), sharex=False)
for ax, p in zip(ejes, ["Café", "Papa"]):
    r = rob[rob.Producto == p].reset_index(drop=True)
    yy = np.arange(len(r))[::-1]
    ax.axvline(0, color="#8a8984", lw=0.9)
    ax.errorbar(r.beta, yy, xerr=1.96 * r.ee, fmt="o", color=COLOR[p], ms=5.5, lw=1.6, capsize=0)
    ax.set_yticks(yy, r["Especificación"], fontsize=7.2)
    ax.set_title(f"{'a' if p == 'Café' else 'b'}) {p}: elasticidad precio de la oferta (IC 95 %)")
    ax.grid(axis="y", visible=False)
    if p == "Papa":
        ax.set_yticklabels([])
fig.tight_layout()
guardar_fig(fig, "Figura_4_robustez")

## 7. Contexto departamental: ingreso bruto y valor agregado agropecuario (hipótesis principal)
Tabla 8, panel B: raíces unitarias de los precios mensuales (ADF y KPSS). Figura 5: peso del ingreso bruto de café y papa frente al VA agropecuario y co-movimiento anual.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
def raiz(s, nombre):
    s = np.log(s.dropna())
    a, k = adfuller(s, autolag="AIC", regression="c"), kpss(s, regression="c", nlags="auto")
    a2 = adfuller(s.diff().dropna(), autolag="AIC", regression="c")
    return (nombre, len(s), a[0], a[1], k[0], k[1], a2[0], a2[1])
serie_cafe = cafe_m[cafe_m.anio.between(ANIO_INI, ANIO_FIN)].set_index("Fecha").Precio
serie_papa = papa_m[(papa_m.plaza == "Pasto") & (papa_m.producto == "Papa") & papa_m.anio.between(ANIO_INI, ANIO_FIN)].set_index("Fecha").Precio
serie_crio = papa_m[(papa_m.plaza == "Pasto") & (papa_m.producto == "Papa criolla") & papa_m.anio.between(ANIO_INI, ANIO_FIN)].set_index("Fecha").Precio
tabla8b = pd.DataFrame([raiz(serie_cafe, "ln precio café (mensual)"), raiz(serie_papa, "ln precio papa (mensual)"),
                        raiz(serie_crio, "ln precio papa criolla (mensual)")],
                       columns=["Serie", "T", "ADF nivel", "p ADF", "KPSS nivel", "p KPSS", "ADF diferencia", "p ADF dif."]).set_index("Serie")
guardar_tabla(tabla8b.round(3), "Tabla_8b_raices_unitarias")
display(tabla8b.round(3))

ctx = dep[dep.anio.between(ANIO_INI, ANIO_FIN)].pivot(index="anio", columns="producto", values="ib_nom") / 1000
ctx["total"] = ctx[["Café", "Papa", "Papa criolla"]].sum(axis=1, min_count=3)
ctx = ctx.join(va.set_index("anio")[["va_nom", "va_real"]])
ctx["peso"] = 100 * ctx.total / ctx.va_nom
for c in ["total", "va_nom", "Café", "Papa"]:
    ctx[f"d_{c}"] = np.log(ctx[c]).diff() * 100
ctx.loc[2019, ["d_total", "d_Papa", "d_Café"]] = np.nan          # quiebre EVA
cc = ctx.dropna(subset=["d_total", "d_va_nom"])
reg_ctx = sm.OLS(cc.d_va_nom, sm.add_constant(cc[["d_total"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 1})
corr_ctx = cc[["d_va_nom", "d_total"]].corr().iloc[0, 1]
print(f"Δln VA nominal vs Δln ingreso bruto café+papa: pendiente {reg_ctx.params['d_total']:.3f} (p = {reg_ctx.pvalues['d_total']:.3f}), "
      f"correlación {corr_ctx:.2f}, n = {int(reg_ctx.nobs)}")

fig, ejes = plt.subplots(1, 2, figsize=(9.4, 3.1))
ax = ejes[0]
anios_c = ctx.index[ctx.total.notna()]
fondo = np.zeros(len(anios_c))
for p in ["Café", "Papa", "Papa criolla"]:
    v = (100 * ctx.loc[anios_c, p] / ctx.loc[anios_c, "va_nom"]).values
    ax.bar(anios_c, v, bottom=fondo, color=COLOR[p], width=0.7, label=p, edgecolor="white", linewidth=0.6)
    fondo += v
ax.set_title("a) Ingreso bruto (precio mayorista) / VA agropecuario (%)")
ax.legend(fontsize=7, ncol=3, loc="upper left")
ax.set_ylim(0, fondo.max() * 1.25)
ax.grid(axis="x", visible=False)
ax = ejes[1]
ax.scatter(cc.d_total, cc.d_va_nom, color=C_VA, s=30, zorder=3)
for a, r in cc.iterrows():
    ax.annotate(str(a), (r.d_total, r.d_va_nom), xytext=(4, 2), textcoords="offset points", fontsize=6.5, color=C_TXT)
xx = np.linspace(cc.d_total.min(), cc.d_total.max(), 20)
ax.plot(xx, reg_ctx.params["const"] + reg_ctx.params["d_total"] * xx, color=C_PAPA, lw=1.4,
        label=f"pendiente {reg_ctx.params['d_total']:.2f} (p = {reg_ctx.pvalues['d_total']:.2f})")
ax.set_xlabel("Δln ingreso bruto café + papa (nominal, %)")
ax.set_ylabel("Δln VA agropecuario nominal (%)")
ax.set_title("b) Co-movimiento anual")
ax.legend(fontsize=7)
fig.tight_layout()
guardar_fig(fig, "Figura_5_contexto_va")

## 8. Cifras clave (se citan en el PDF)

In [ ]:
def ib_share(p, nivel):
    return float(tabla6.loc[(p, nivel), "Precio"])
mods = {k: (float(v.params["ln_p_real_l1"]), float(v.pvalues["ln_p_real_l1"])) for k, v in MODELOS.items()}
CLAVE = {
    "n_mpios": int(est.cod_mpio.nunique()), "n_obs": int(len(est)),
    "s_precio_cafe_dep": ib_share("Café", "Departamento"), "s_precio_papa_dep": ib_share("Papa", "Departamento"),
    "s_precio_criolla_dep": ib_share("Papa criolla", "Departamento"),
    "s_precio_cafe_mun": ib_share("Café", "Municipios (panel)"), "s_precio_papa_mun": ib_share("Papa", "Municipios (panel)"),
    "s_precio_criolla_mun": ib_share("Papa criolla", "Municipios (panel)"),
    "s_rend_cafe_mun": float(tabla6.loc[("Café", "Municipios (panel)"), "Rendimiento"]),
    "s_rend_papa_mun": float(tabla6.loc[("Papa", "Municipios (panel)"), "Rendimiento"]),
    "s_area_cafe_mun": float(tabla6.loc[("Café", "Municipios (panel)"), "Área"]),
    "s_area_papa_mun": float(tabla6.loc[("Papa", "Municipios (panel)"), "Área"]),
    "beta_cafe_area": mods["Café · área"][0], "p_cafe_area": mods["Café · área"][1],
    "beta_cafe_prod": mods["Café · producción"][0], "p_cafe_prod": mods["Café · producción"][1],
    "beta_papa_area": mods["Papa · área"][0], "p_papa_area": mods["Papa · área"][1],
    "beta_papa_prod": mods["Papa · producción"][0], "p_papa_prod": mods["Papa · producción"][1],
    "beta_criolla_area": mods["Papa criolla · área"][0], "p_criolla_area": mods["Papa criolla · área"][1],
    "ctx_pendiente": float(reg_ctx.params["d_total"]), "ctx_p": float(reg_ctx.pvalues["d_total"]),
    "ctx_corr": float(corr_ctx), "ctx_n": int(reg_ctx.nobs),
    "peso_min": float(ctx.peso.min()), "peso_max": float(ctx.peso.max()), "peso_medio": float(ctx.peso.mean()),
    "vol_precio_cafe": float(tabla5.loc["Volatilidad mensual del precio (desv. Δln, %)", "Café"]),
    "vol_precio_papa": float(tabla5.loc["Volatilidad mensual del precio (desv. Δln, %)", "Papa"]),
}
for _, r in acum.iterrows():
    k = f"acum_{norm(r.Producto).lower().replace(' ', '_')}_{r.Tramo[:4]}"
    for comp in ["Área", "Rendimiento", "Precio", "Total"]:
        CLAVE[f"{k}_{norm(comp).lower()}"] = float(r[comp])
(BASE / "salidas" / "resultados_clave.json").write_text(json.dumps(CLAVE, indent=2))
display(pd.Series(CLAVE).round(3).to_frame("valor"))

## 9. Aplicación interactiva
Ejecute la celda: aparece un tablero con **filtros a la izquierda** (cultivo, municipio, años, tipo de precio y plaza de referencia de la papa), **indicadores** que cambian con los filtros y **pestañas**: Panorama · Mapa · Descomposición · Modelo · Datos.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import ipywidgets as W
pio.renderers.default = "colab" if EN_COLAB else "notebook_connected"
if EN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

GEO_N = json.loads((RAW / "mapa_municipios_narino.geojson").read_text(encoding="utf-8"))
PANELES = {"Pasto": panel, "Ipiales": panel_ipiales}
A_MIN, A_MAX = 2012, 2024

def datos_app(cultivo, mpio, a0, a1, plaza="Pasto"):
    """Serie anual (Nariño o un municipio) del cultivo elegido."""
    d = PANELES[plaza]
    d = d[(d.producto == cultivo) & d.anio.between(a0, a1)]
    if mpio != "TODOS":
        d = d[d.cod_mpio == mpio]
    s = d.groupby("anio").agg(prod=("prod", lambda x: x.sum(min_count=1)), area_cos=("area_cos", lambda x: x.sum(min_count=1)),
                              p_nom=("p_nom", "first"), p_real=("p_real", "first"))
    s["rend"] = s["prod"] / s.area_cos
    for t in ("nom", "real"):
        s[f"ib_{t}"] = s["prod"] * s[f"p_{t}"] / 1e3          # millones de pesos
    return s.reset_index()

def kpis_html(s, cultivo, real):
    t = "real" if real else "nom"
    ib = s[f"ib_{t}"].dropna()
    var = 100 * (ib.iloc[-1] / ib.iloc[0] - 1) if len(ib) > 1 else np.nan
    tarjetas = [("Producción media", f"{s['prod'].mean():,.0f} t"), ("Área cosechada media", f"{s.area_cos.mean():,.0f} ha"),
                ("Rendimiento medio", f"{s.rend.mean():,.2f} t/ha"),
                (f"Precio medio ({'$ 2015' if real else 'corriente'})", f"${s[f'p_{t}'].mean():,.0f}/kg"),
                ("Ingreso bruto medio", f"${ib.mean() / 1e3:,.1f} mil millones"),
                ("Variación del ingreso bruto", f"{var:+.0f} %" if pd.notna(var) else "—")]
    color = COLOR[cultivo]
    cajas = "".join(f"<div class='kpi'><div class='kpi-t'>{a}</div><div class='kpi-v'>{b}</div></div>" for a, b in tarjetas)
    return f"<div class='kpis' style='--c:{color}'>{cajas}</div>"

def fig_panorama(s, cultivo, real):
    t = "real" if real else "nom"
    f = make_subplots(rows=1, cols=3, subplot_titles=("Producción (t)", f"Precio ({'$ de 2015' if real else '$ corrientes'}/kg)",
                                                      "Ingreso bruto (millones de $)"), horizontal_spacing=0.07)
    c = COLOR[cultivo]
    f.add_bar(x=s.anio, y=s["prod"], marker_color=c, row=1, col=1, name="Producción", hovertemplate="%{x}: %{y:,.0f} t<extra></extra>")
    f.add_scatter(x=s.anio, y=s[f"p_{t}"], mode="lines+markers", line=dict(color=c, width=2.5), row=1, col=2, name="Precio",
                  hovertemplate="%{x}: $%{y:,.0f}/kg<extra></extra>")
    f.add_scatter(x=s.anio, y=s[f"ib_{t}"], mode="lines+markers", line=dict(color="#4b4a46", width=2.5), row=1, col=3,
                  name="Ingreso bruto", hovertemplate="%{x}: $%{y:,.0f} millones<extra></extra>")
    if s.anio.min() <= 2018 < s.anio.max():
        f.add_vline(x=2018.5, line_dash="dot", line_color="#8a8984", row=1, col=1)
    f.update_layout(template="simple_white", height=380, showlegend=False, margin=dict(l=40, r=20, t=50, b=30))
    return f

METRICAS = {"Producción (t)": "prod", "Área cosechada (ha)": "area_cos", "Rendimiento (t/ha)": "rend",
            "Ingreso bruto real (millones $ 2015)": "ib_real", "Participación del precio en la varianza del ingreso": "s_precio"}

def fig_mapa(cultivo, a0, a1, metrica, plaza="Pasto"):
    d = PANELES[plaza]
    d = d[(d.producto == cultivo) & d.anio.between(a0, a1)]
    if metrica == "s_precio":
        v = (d.dropna(subset=["d_ib_real"] + list(COMP)).groupby("cod_mpio").filter(lambda g: len(g) >= 4)
             .groupby("cod_mpio").apply(lambda g: participaciones(g)["Precio"]).rename("valor").reset_index())
    else:
        v = d.groupby("cod_mpio")[metrica].mean().rename("valor").reset_index()
        if metrica == "ib_real":
            v["valor"] = v.valor
    v["municipio"] = v.cod_mpio.map(NOMBRE_MPIO)
    nombre = next(k for k, x in METRICAS.items() if x == metrica)
    f = px.choropleth(v, geojson=GEO_N, locations="cod_mpio", featureidkey="properties.MPIO_CCNCT", color="valor",
                      hover_name="municipio", color_continuous_scale="YlOrBr" if cultivo == "Café" else "Blues",
                      labels={"valor": nombre})
    f.update_geos(fitbounds="locations", visible=False)
    f.update_layout(height=520, margin=dict(l=0, r=0, t=30, b=0), title=f"{cultivo} · {nombre} · {a0}–{a1}")
    return f

def fig_descomp(s, cultivo):
    s = s.set_index("anio")
    validos = s.dropna(subset=["prod", "p_real"]).index
    if len(validos) < 2:
        return None, "No hay suficientes años con producción y precio."
    a0, a1 = validos.min(), validos.max()
    comp = {"Área": np.log(s.area_cos[a1] / s.area_cos[a0]) * 100, "Rendimiento": np.log(s.rend[a1] / s.rend[a0]) * 100,
            "Precio": np.log(s.p_real[a1] / s.p_real[a0]) * 100}
    total = sum(comp.values())
    f = go.Figure(go.Waterfall(x=list(comp) + ["Ingreso bruto"], y=list(comp.values()) + [total],
                               measure=["relative"] * 3 + ["total"], connector={"line": {"color": "#b9b8b2"}},
                               increasing={"marker": {"color": "#2a78d6"}}, decreasing={"marker": {"color": "#eb6834"}},
                               totals={"marker": {"color": "#4b4a46"}}, texttemplate="%{y:+.1f}", textposition="outside"))
    f.update_layout(template="simple_white", height=380, title=f"Cambio acumulado {a0}–{a1} (100·Δln, precios reales)",
                    margin=dict(l=40, r=20, t=50, b=30))
    nota = "⚠️ El rango cruza el cambio metodológico de la EVA (2019): el componente de cantidad incluye ese salto." if a0 <= 2018 < a1 else ""
    return f, nota

def tabla_modelo(cultivo, y, rezago, d2019, sin_covid, errores, plaza):
    d = PANELES[plaza]
    d = d[(d.producto == cultivo) & (d["prod"] > 0) & d.anio.between(ANIO_INI, ANIO_FIN)]
    if sin_covid:
        d = d[~d.anio.isin([2020, 2021])]
    ctrl = ("tendencia", "D2019") if d2019 else ("tendencia",)
    r = estimar_oferta(d, y, f"ln_p_real_l{rezago}", ctrl, "kernel" if errores == "Driscoll-Kraay" else "cluster")
    t = pd.DataFrame({"coeficiente": r.params, "error estándar": r.std_errors, "valor p": r.pvalues}).round(3)
    t.index = [f"ln precio real (t−{rezago})" if i.startswith("ln_p") else ETQ.get(i, i) for i in t.index]
    return t, r

CSS = """<style>
.app-head{background:linear-gradient(90deg,#3b2a1a,#2a78d6);color:#fff;padding:14px 18px;border-radius:10px;margin-bottom:10px}
.app-head h2{margin:0;font-size:20px}.app-head p{margin:4px 0 0;font-size:12px;opacity:.9}
.kpis{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin:4px 0 10px}
.kpi{background:#fff;border:1px solid #e3e1da;border-left:5px solid var(--c);border-radius:8px;padding:8px 10px}
.kpi-t{font-size:11px;color:#6b6a65}.kpi-v{font-size:18px;font-weight:700;color:#222}
.side-t{font-weight:700;font-size:13px;margin:6px 0 2px;color:#3b2a1a}
</style>"""

def lanzar_app():
    opciones_mpio = lambda c: [("Todo Nariño", "TODOS")] + sorted(
        [(NOMBRE_MPIO[m], m) for m in panel[(panel.producto == c) & (panel["prod"] > 0)].cod_mpio.unique()])
    w_cult = W.ToggleButtons(options=["Café", "Papa", "Papa criolla"], value="Papa", style={"button_width": "78px"})
    w_mpio = W.Dropdown(options=opciones_mpio("Papa"), value="TODOS", layout=W.Layout(width="230px"))
    w_anios = W.IntRangeSlider(value=(ANIO_INI, ANIO_FIN), min=A_MIN, max=A_MAX, continuous_update=False, layout=W.Layout(width="235px"))
    w_real = W.ToggleButtons(options=[("Reales", True), ("Nominales", False)], value=True, style={"button_width": "78px"})
    w_plaza = W.Dropdown(options=["Pasto", "Ipiales"], value="Pasto", layout=W.Layout(width="230px"))
    w_metrica = W.Dropdown(options=METRICAS, value="prod", description="Variable:", layout=W.Layout(width="440px"))
    w_y = W.Dropdown(options=[("Área cosechada", "ln_area_cos"), ("Producción", "ln_prod"), ("Área sembrada", "ln_area_sem")],
                     value="ln_area_cos", description="Dependiente:")
    w_rez = W.ToggleButtons(options=[("t−1", 1), ("t−2", 2)], value=1, description="Rezago:")
    w_d19 = W.Checkbox(value=True, description="Dummy cambio EVA 2019")
    w_cov = W.Checkbox(value=False, description="Excluir 2020–2021")
    w_err = W.ToggleButtons(options=["Driscoll-Kraay", "Agrupados"], value="Driscoll-Kraay", description="Errores:")
    w_desc = W.Button(description="Descargar CSV", icon="download", button_style="primary")

    o_kpi, o_pan, o_map, o_dec, o_mod, o_dat = (W.Output() for _ in range(6))
    lado = W.VBox([W.HTML("<div class='side-t'>Cultivo</div>"), w_cult, W.HTML("<div class='side-t'>Municipio</div>"), w_mpio,
                   W.HTML("<div class='side-t'>Años</div>"), w_anios, W.HTML("<div class='side-t'>Precios</div>"), w_real,
                   W.HTML("<div class='side-t'>Plaza de referencia (papa)</div>"), w_plaza,
                   W.HTML("<p style='font-size:11px;color:#6b6a65;margin-top:12px'>Fuentes: EVA (MinAgricultura–UPRA), SIPSA-DANE, "
                          "FNC y cuentas departamentales del DANE. Precios reales con el deflactor del VA agropecuario de Nariño.</p>")],
                  layout=W.Layout(width="265px", padding="10px", border="1px solid #e3e1da", border_radius="10px"))
    pestanas = W.Tab(children=[o_pan, W.VBox([w_metrica, o_map]), o_dec,
                               W.VBox([W.HBox([w_y, w_rez]), W.HBox([w_d19, w_cov]), w_err, o_mod]), W.VBox([w_desc, o_dat])])
    for i, t in enumerate(["Panorama", "Mapa", "Descomposición", "Modelo", "Datos"]):
        pestanas.set_title(i, t)
    principal = W.VBox([o_kpi, pestanas], layout=W.Layout(width="100%", padding="0 0 0 12px"))

    def estado():
        a0, a1 = w_anios.value
        return w_cult.value, w_mpio.value, a0, a1, w_real.value, w_plaza.value

    def refrescar(*_):
        c, m, a0, a1, real, plaza = estado()
        s = datos_app(c, m, a0, a1, plaza)
        with o_kpi:
            o_kpi.clear_output(wait=True)
            display(HTML(kpis_html(s, c, real)))
        with o_pan:
            o_pan.clear_output(wait=True)
            fig_panorama(s, c, real).show()
        with o_map:
            o_map.clear_output(wait=True)
            fig_mapa(c, a0, a1, w_metrica.value, plaza).show()
        with o_dec:
            o_dec.clear_output(wait=True)
            f, nota = fig_descomp(s, c)
            if f is not None:
                f.show()
            if nota:
                print(nota)
        with o_dat:
            o_dat.clear_output(wait=True)
            display(s.round(2))
        refrescar_modelo()

    def refrescar_modelo(*_):
        c, _, _, _, _, plaza = estado()
        with o_mod:
            o_mod.clear_output(wait=True)
            try:
                t, r = tabla_modelo(c, w_y.value, w_rez.value, w_d19.value, w_cov.value, w_err.value, plaza)
                display(HTML(f"<b>{c}</b> · efectos fijos de municipio · {int(r.nobs)} observaciones · "
                             f"{r.entity_info['total']:.0f} municipios · R² within {r.rsquared_within:.3f}"))
                display(t)
                b = t.iloc[0]
                print(f"Elasticidad: un aumento de 10 % en el precio real se asocia con {10 * b['coeficiente']:+.1f} % en la variable dependiente.")
            except Exception as e:
                print("No se pudo estimar con esta combinación:", type(e).__name__)

    def cambiar_cultivo(ch):
        w_mpio.options = opciones_mpio(ch["new"])
        w_mpio.value = "TODOS"

    def descargar_csv(_):
        c, m, a0, a1, real, plaza = estado()
        ruta = TAB / f"app_{norm(c).lower().replace(' ', '_')}_{m}_{a0}_{a1}.csv"
        datos_app(c, m, a0, a1, plaza).to_csv(ruta, index=False, encoding="utf-8-sig")
        if EN_COLAB:
            from google.colab import files
            files.download(str(ruta))
        with o_dat:
            print("Guardado en", ruta)

    w_cult.observe(cambiar_cultivo, "value")
    for w in (w_cult, w_mpio, w_anios, w_real, w_plaza, w_metrica):
        w.observe(refrescar, "value")
    for w in (w_y, w_rez, w_d19, w_cov, w_err):
        w.observe(refrescar_modelo, "value")
    w_desc.on_click(descargar_csv)
    display(HTML(CSS + "<div class='app-head'><h2>Café y papa en Nariño · tablero</h2>"
                       "<p>Producción, precios e ingreso bruto 2012–2024 · ¿qué mueve el ingreso: cantidades o precios?</p></div>"))
    display(W.HBox([lado, principal]))
    refrescar()

lanzar_app()

## 10. Exportar resultados
Empaqueta `salidas/` (figuras y tablas con la numeración del PDF) en `salidas.zip`; en Colab se descarga automáticamente.

In [ ]:
ruta_zip = BASE / "salidas.zip"
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted((BASE / "salidas").rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(BASE))
print("Archivos generados:")
for f in sorted((BASE / "salidas").rglob("*.*")):
    print("  ", f.relative_to(BASE))
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))